In [3]:
from utils import *
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering, KMeans
from scipy.cluster.hierarchy import linkage
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [16]:
df = pd.read_csv('C:\\Users\\windows\\.cache\\kagglehub\\datasets\\amar5693\\retail-business-analytics-dataset-10k-orders\\versions\\1\\Business_Analytics_Dataset_10000_Rows.csv')
df["Order_Date"] = pd.to_datetime(df["Order_Date"])
df["year"] = df["Order_Date"].dt.year
df["month"] = df["Order_Date"].dt.month
df["day"] = df["Order_Date"].dt.day
df["weekday"] = df["Order_Date"].dt.weekday  # 0=Lunes, 6=Domingo
df.head()

,Order_ID,Customer_ID,Order_Date,Region,Product_Category,Customer_Segment,Quantity,Unit_Price,Discount_Rate,Revenue,Cost,Profit,Payment_Method,year,month,day,weekday
0,1,CUST3818,2024-08-18,North,Clothing,Corporate,5,300.68,0.27,1097.48,768.29,329.19,Credit Card,2024,8,18,6
1,2,CUST9689,2024-06-19,South,Beauty,Home Office,9,32.89,0.02,290.09,179.33,110.76,Debit Card,2024,6,19,2
2,3,CUST9147,2024-11-21,West,Sports,Corporate,5,345.61,0.25,1296.04,1022.60,273.44,Credit Card,2024,11,21,3
3,4,CUST7938,2024-07-19,North,Clothing,Consumer,1,444.50,0.06,417.83,280.99,136.84,UPI,2024,7,19,4
4,5,CUST5127,2024-10-28,South,Home & Kitchen,Consumer,5,65.13,0.21,257.26,151.90,105.36,Credit Card,2024,10,28,0


In [17]:

df.drop(['Order_ID','Order_Date','Customer_ID'], axis=1, inplace=True)
# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Apply OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, drop='first')
df_encoded = pd.DataFrame(
    encoder.fit_transform(df[categorical_cols]),
    columns=encoder.get_feature_names_out(categorical_cols)
)

# Concatenate encoded columns with numerical columns
df = pd.concat([df.drop(categorical_cols, axis=1).reset_index(drop=True), df_encoded], axis=1)
df.head()

C:\Users\windows\AppData\Local\Temp\ipykernel_16404\4149562937.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns.tolist()


,Quantity,Unit_Price,Discount_Rate,Revenue,Cost,Profit,year,month,day,weekday,...,Product_Category_Clothing,Product_Category_Electronics,Product_Category_Home & Kitchen,Product_Category_Sports,Customer_Segment_Corporate,Customer_Segment_Home Office,Payment_Method_Credit Card,Payment_Method_Debit Card,Payment_Method_Net Banking,Payment_Method_UPI
0,5,300.68,0.27,1097.48,768.29,329.19,2024,8,18,6,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1,9,32.89,0.02,290.09,179.33,110.76,2024,6,19,2,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
2,5,345.61,0.25,1296.04,1022.60,273.44,2024,11,21,3,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
3,1,444.50,0.06,417.83,280.99,136.84,2024,7,19,4,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,65.13,0.21,257.26,151.90,105.36,2024,10,28,0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [ ]:
# SEMILLA para reproducibilidad
SEMILLA = 666


X = df.copy()
feature_names = df.columns.tolist()

# 2) Escalar
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

# Para visualizar en 2D
idx_a, idx_b = 0, 1
X2 = Xs[:, [idx_a, idx_b]]
xlab = f"{feature_names[idx_a]} (scaled)"
ylab = f"{feature_names[idx_b]} (scaled)"

# -------------------------
# A) Clustering Jerárquico
# -------------------------

Z = linkage(Xs, method="ward")
hier = AgglomerativeClustering(n_clusters=3, linkage="ward")
labels_h = hier.fit_predict(Xs)

evaluate_clustering(Xs, labels_h, name="Hierarchical (Agglomerative, ward, k=3)")

# -----------
# B) K-Means
# -----------
kmeans = KMeans(n_clusters=3, init="k-means++", n_init=10, random_state=SEMILLA)
labels_k = kmeans.fit_predict(Xs)

evaluate_clustering(Xs, labels_k, name="K-Means (k=3, k-means++, n_init=10)")

# -------------
# C) Visualización comparativa en un solo lienzo
# -------------
fig, axs = plt.subplots(2, 2, figsize=(12, 10))

# dendrograma
from scipy.cluster.hierarchy import dendrogram

dendrogram(Z, truncate_mode="level", p=5, ax=axs[0, 0])
axs[0, 0].set_title("Dendrograma (ward)")
axs[0, 0].set_xlabel("Índices / clusters combinados")
axs[0, 0].set_ylabel("Distancia")

cmap = plt.cm.rainbow
norm = plt.Normalize(vmin=0, vmax=2)
axs[0, 1].scatter(X2[:, 0], X2[:, 1], c=labels_h, s=35, cmap=cmap, norm=norm)
axs[0, 1].set_title("Hierarchical k=3")
axs[0, 1].set_xlabel(xlab)
axs[0, 1].set_ylabel(ylab)

# k-means (2D) con mismo esquema de colores
axs[1, 0].scatter(X2[:, 0], X2[:, 1], c=labels_k, s=35, cmap=cmap, norm=norm)
axs[1, 0].set_title("K-Means k=3")
axs[1, 0].set_xlabel(xlab)
axs[1, 0].set_ylabel(ylab)

# clases reales (2D) igual
axs[1, 1].scatter(X2[:, 0], X2[:, 1], c=y_true, s=35, cmap=cmap, norm=norm)
axs[1, 1].set_title("Clases reales")
axs[1, 1].set_xlabel(xlab)
axs[1, 1].set_ylabel(ylab)

if class_names is not None:
    handles = []
    for lbl in np.unique(y_true):
        handles.append(plt.Line2D([0], [0], marker="o", color="w",
                                  markerfacecolor=plt.cm.rainbow(lbl / len(class_names)),
                                  markersize=6))
    axs[1, 1].legend(handles, class_names, title="Clases", loc="best")

plt.tight_layout()
plt.show()